# Official CHESCA vs CHESCA-based Peer Mesh

이 notebook은 내려받은 공식 `CHESCA-main`을 그대로 실행하는 기준선과, 공식 CHESCA 위에 분산 배터리 협상을 추가한 mesh를 같은 CityLearn schema에서 비교합니다.

## 1. Google Drive 연결

`chesca_vs_mesh` 폴더 전체를 `MyDrive` 바로 아래에 올린 뒤 실행합니다.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/chesca_vs_mesh')
OFFICIAL_DIR = PROJECT_DIR / 'CHESCA-main'
assert (OFFICIAL_DIR / 'checa' / 'agent.py').exists(), f'공식 CHESCA 폴더를 찾을 수 없습니다: {OFFICIAL_DIR}'
print('Project:', PROJECT_DIR)
print('Official source:', OFFICIAL_DIR)

Mounted at /content/drive
Project: /content/drive/MyDrive/chesca_vs_mesh
Official source: /content/drive/MyDrive/chesca_vs_mesh/CHESCA-main


## 2. 설치

공식 README가 요구하는 `CityLearn==2.1b12`의 Python 런타임은 이 폴더의 `third_party/CityLearn-2.1b12`에 포함되어 있습니다. 오래된 PyPI 설치가 Colab의 NumPy/Pandas/SciPy 스택을 깨뜨리지 않도록 CityLearn 자체는 pip로 다시 설치하지 않습니다.

In [2]:
%pip install -q "gym==0.26.2" "simplejson>=3.19" "xgboost>=1.7,<3"

import sys
VENDORED_CITYLEARN = PROJECT_DIR / 'third_party' / 'CityLearn-2.1b12'
assert (VENDORED_CITYLEARN / 'citylearn' / 'citylearn.py').exists(), f'CityLearn runtime을 찾을 수 없습니다: {VENDORED_CITYLEARN}'
sys.path.insert(0, str(VENDORED_CITYLEARN))

import numpy as np
import pandas as pd
import scipy
import torch
import xgboost
import citylearn
from citylearn.citylearn import CityLearnEnv

print('numpy:', np.__version__, 'pandas:', pd.__version__, 'scipy:', scipy.__version__)
print('torch:', torch.__version__, 'xgboost:', xgboost.__version__)
print('CityLearn:', citylearn.__version__, citylearn.__file__)
assert citylearn.__version__ == '2.1b12'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 14.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


numpy: 2.0.2 pandas: 2.2.2 scipy: 1.16.3
torch: 2.11.0+cu128 xgboost: 2.1.4
CityLearn: 2.1b12 /content/drive/MyDrive/chesca_vs_mesh/third_party/CityLearn-2.1b12/citylearn/__init__.py


## 3. 프로젝트 불러오기와 설정

`power_outage_seed=None`은 공식 원본 실행 조건을 유지합니다. 동일한 stochastic outage seed를 고정한 paired 비교가 필요하면 정수로 변경하십시오.

In [3]:
import sys
sys.path.insert(0, str(PROJECT_DIR / 'src'))

from chesca_vs_mesh import BenchmarkSuite, MeshConfig, available_datasets

print('Available bundled datasets:')
print(available_datasets())

DATASET = 'citylearn_challenge_2023_phase_3_1'
EPISODE_STEPS = None  # None: official schema의 전체 기간. 빠른 smoke test는 71 등으로 변경.
TAG = 'official_chesca_vs_peer_mesh'

mesh_config = MeshConfig(
    rounds=3,
    offer_step=0.04,
    target_quantile=0.65,
    peak_weight=1.00,
    ramp_weight=0.32,
    price_weight=0.10,
    carbon_weight=0.08,
)
suite = BenchmarkSuite(
    output_directory=PROJECT_DIR / 'results',
    mesh_config=mesh_config,
    power_outage_seed=None,
)

Available bundled datasets:
['citylearn_challenge_2023_phase_1', 'citylearn_challenge_2023_phase_2_local_evaluation', 'citylearn_challenge_2023_phase_2_online_evaluation_1', 'citylearn_challenge_2023_phase_2_online_evaluation_2', 'citylearn_challenge_2023_phase_2_online_evaluation_3', 'citylearn_challenge_2023_phase_3_1', 'citylearn_challenge_2023_phase_3_2', 'citylearn_challenge_2023_phase_3_3', 'warm_up']


## 4. 공식 CHESCA 재현과 Mesh 비교 실행

`chesca_official` 행은 원본 `CHESCA-main/agents/user_agent.py`의 agent를 수정 없이 실행합니다. `challenge_cost`는 논문의 가중 목적함수이고, 변화율은 공식 CHESCA 대비이며 낮을수록 좋은 KPI에서 음수이면 mesh 개선입니다.

In [4]:
result = suite.compare_controllers(
    dataset_name=DATASET,
    controllers=['chesca_official', 'chesca_mesh'],
    episode_steps=EPISODE_STEPS,
    tag=TAG,
)

display(result.summary)
display(result.citylearn_metrics)
print('Results saved to:', result.output_directory)

,controller,steps,grid_import_kwh,total_cost,carbon_kg,peak_kwh,ramping_kwh,agent_time_seconds,agreement_steps,changed_trade_steps,...,challenge_cost_change_vs_chesca_pct,comfort_cost_change_vs_chesca_pct,emissions_cost_change_vs_chesca_pct,grid_cost_change_vs_chesca_pct,resilience_cost_change_vs_chesca_pct,grid_import_kwh_change_vs_chesca_pct,total_cost_change_vs_chesca_pct,carbon_kg_change_vs_chesca_pct,peak_kwh_change_vs_chesca_pct,ramping_kwh_change_vs_chesca_pct
0,chesca_official,2207,15146.473406,513.101355,6721.283708,30.452171,4226.040989,172.837657,0,0,...,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
1,chesca_mesh,2207,15162.901138,511.475746,6723.459507,29.626965,4076.309230,183.748674,2163,1774,...,-0.181399,0.303553,0.026367,-1.4841,1.782102,0.108459,-0.31682,0.032372,-2.709845,-3.543074


,controller,carbon_emissions_total,discomfort_proportion,ramping_average,daily_one_minus_load_factor_average,daily_peak_average,annual_peak_average,one_minus_thermal_resilience_proportion,power_outage_normalized_unserved_energy_total,average_score
0,chesca_official,0.925583,0.130315,0.850411,0.958645,0.889024,1.132991,0.788781,0.352457,0.590169
1,chesca_mesh,0.925827,0.130710,0.819864,0.960908,0.891153,1.102288,0.805797,0.355779,0.589098


Results saved to: /content/drive/MyDrive/chesca_vs_mesh/results/citylearn_challenge_2023_phase_3_1/official_chesca_vs_peer_mesh


## 5. 소통이 실제 action을 바꿨는지 확인

`negotiations`는 step 단위 선택 결과이고 `messages`는 peer broadcast 기록입니다. 메시지만 많고 `changed_peers`가 거의 없으면 signal이 너무 보수적이고, 변경은 많은데 KPI가 나빠지면 offer 또는 shadow weight를 재설계해야 합니다.

In [5]:
if result.negotiations.empty:
    print('Mesh negotiation log가 없습니다.')
else:
    display(result.negotiations.describe(include='all'))
    display(result.negotiations.tail(20))

if not result.messages.empty:
    display(result.messages.tail(20))

,controller,step,hour,active_peers,changed_peers,official_predicted_grid,negotiated_predicted_grid,predicted_grid_delta,district_target,final_shadow_signal,logical_message_count
count,2163,2163.000000,2163.000000,2163.0,2163.000000,2163.000000,2163.000000,2163.000000,2163.000000,2163.000000,2163.0
unique,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,chesca_mesh,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,2163,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,1097.150717,12.469718,6.0,4.807675,7.229408,6.910483,-0.318925,8.168960,0.195643,90.0
std,NaN,641.588071,6.929297,0.0,2.336414,3.302388,3.101416,0.652244,0.960874,0.501341,0.0
min,NaN,1.000000,1.000000,6.0,0.000000,-0.520464,-0.238569,-0.876000,4.711030,-1.585736,90.0
25%,NaN,541.500000,6.000000,6.0,6.000000,4.904664,4.583329,-0.876000,7.632508,-0.095382,90.0
50%,NaN,1082.000000,12.000000,6.0,6.000000,6.702462,6.676038,-0.876000,8.316833,0.111395,90.0
75%,NaN,1665.500000,18.000000,6.0,6.000000,9.459188,9.029656,0.220720,8.889881,0.340816,90.0


,controller,step,hour,active_peers,changed_peers,official_predicted_grid,negotiated_predicted_grid,predicted_grid_delta,district_target,final_shadow_signal,logical_message_count
2143,chesca_mesh,2187,4,6,6,7.136674,8.012674,8.760000e-01,9.463839,-0.379035,90
2144,chesca_mesh,2188,5,6,6,6.004149,6.727995,7.238458e-01,9.463839,-0.109891,90
2145,chesca_mesh,2189,6,6,0,6.134526,6.134526,-8.881784e-16,9.463839,0.003593,90
2146,chesca_mesh,2190,7,6,6,5.427373,5.886944,4.595713e-01,9.463839,-0.152118,90
2147,chesca_mesh,2191,8,6,0,3.990087,3.990087,0.000000e+00,9.463839,-0.059282,90
2148,chesca_mesh,2192,9,6,0,3.673563,3.673563,0.000000e+00,9.463839,0.039845,90
2149,chesca_mesh,2193,10,6,6,2.975905,3.225821,2.499158e-01,9.463839,-0.111892,90
2150,chesca_mesh,2194,11,6,0,2.704166,2.704166,0.000000e+00,9.463839,0.007292,90
2151,chesca_mesh,2195,12,6,0,4.576346,4.576346,0.000000e+00,9.463839,0.060608,90
2152,chesca_mesh,2196,13,6,6,5.933519,5.105570,-8.279485e-01,9.463839,0.189153,90


,controller,step,round_id,sender,official_grid,proposed_grid,lower_grid,upper_grid,soc,district_proposal,district_target,shadow_signal,recipient_count
38914,chesca_mesh,2205,2,4,3.171545,3.171545,3.011545,3.331545,0.559939,10.345062,9.417259,-0.094209,5
38915,chesca_mesh,2205,2,5,1.730219,1.730219,1.598219,1.862219,0.559052,10.345062,9.417259,-0.094209,5
38916,chesca_mesh,2206,0,0,1.546609,1.546609,1.386609,1.706609,0.588083,10.378746,9.417259,0.023331,5
38917,chesca_mesh,2206,0,1,0.997423,0.997423,0.837423,1.157423,0.588083,10.378746,9.417259,0.023331,5
38918,chesca_mesh,2206,0,2,1.137584,1.137584,1.005584,1.269584,0.587355,10.378746,9.417259,0.023331,5
38919,chesca_mesh,2206,0,3,2.078996,2.078996,1.946996,2.210996,0.587355,10.378746,9.417259,0.023331,5
38920,chesca_mesh,2206,0,4,2.833244,2.833244,2.673244,2.993244,0.588083,10.378746,9.417259,0.023331,5
38921,chesca_mesh,2206,0,5,1.784891,1.784891,1.652891,1.916891,0.587355,10.378746,9.417259,0.023331,5
38922,chesca_mesh,2206,1,0,1.546609,1.546609,1.386609,1.706609,0.588083,10.378746,9.417259,0.023331,5
38923,chesca_mesh,2206,1,1,0.997423,0.997423,0.837423,1.157423,0.588083,10.378746,9.417259,0.023331,5


## 6. 논문형 Public Cost / Private Cost 계산

논문 Section 5.1의 비용식은 `0.30 comfort + 0.10 emissions + 0.30 grid + 0.30 resilience`입니다. Public Cost는 3개 건물 public schema 세 실행, Private Cost는 6개 건물 private schema 세 실행의 평균으로 집계합니다. 이 셀은 공식 CHESCA와 mesh 각각 6회씩 전체 기간을 실행하므로 시간이 걸립니다.

In [6]:
leaderboard = suite.compare_public_private_costs(
    controllers=['chesca_official', 'chesca_mesh'],
    episode_steps=None,
    tag='paper_public_private_cost',
)

display(leaderboard.paper_table)
display(leaderboard.summary)
display(leaderboard.runs[['split', 'run_id', 'dataset', 'controller', 'challenge_cost']])
print('Public/private results saved to:', leaderboard.output_directory)

,controller,Private Cost,Public Cost,Private Cost Change vs CHESCA (%),Public Cost Change vs CHESCA (%)
0,chesca_mesh,0.566371,0.502961,-0.239082,-1.071209
1,chesca_official,0.567729,0.508408,0.000000,0.000000


,split,controller,leaderboard_cost,comfort_cost,emissions_cost,grid_cost,resilience_cost,leaderboard_cost_change_vs_chesca_pct
0,private,chesca_mesh,0.566371,0.129195,0.928669,0.920573,0.528580,-0.239082
1,private,chesca_official,0.567729,0.128942,0.928365,0.934023,0.520009,0.000000
2,public,chesca_mesh,0.502961,0.070449,0.951868,0.879749,0.409050,-1.071209
3,public,chesca_official,0.508408,0.070520,0.951344,0.888128,0.418929,0.000000


,split,run_id,dataset,controller,challenge_cost
0,public,1,citylearn_challenge_2023_phase_2_online_evalua...,chesca_official,0.536594
1,public,1,citylearn_challenge_2023_phase_2_online_evalua...,chesca_mesh,0.533249
2,public,2,citylearn_challenge_2023_phase_2_online_evalua...,chesca_official,0.520862
3,public,2,citylearn_challenge_2023_phase_2_online_evalua...,chesca_mesh,0.518596
4,public,3,citylearn_challenge_2023_phase_2_online_evalua...,chesca_official,0.467766
5,public,3,citylearn_challenge_2023_phase_2_online_evalua...,chesca_mesh,0.457040
6,private,1,citylearn_challenge_2023_phase_3_1,chesca_official,0.590169
7,private,1,citylearn_challenge_2023_phase_3_1,chesca_mesh,0.589098
8,private,2,citylearn_challenge_2023_phase_3_2,chesca_official,0.553444
9,private,2,citylearn_challenge_2023_phase_3_2,chesca_mesh,0.549014


Public/private results saved to: /content/drive/MyDrive/chesca_vs_mesh/results/paper_public_private_cost


## 7. 선택적 다중 schema 요약 검증

아래 셀은 `RUN_GENERALIZATION=True`로 바꿨을 때만 실행됩니다. 공식 CHESCA와 mesh를 phase 3의 별도 schema에서도 비교해 특정 한 데이터 구간에만 맞춘 개선인지 점검합니다.

In [7]:
RUN_GENERALIZATION = False

if RUN_GENERALIZATION:
    tables = []
    for dataset in [
        'citylearn_challenge_2023_phase_3_1',
        'citylearn_challenge_2023_phase_3_2',
        'citylearn_challenge_2023_phase_3_3',
    ]:
        evaluated = suite.compare_controllers(
            dataset_name=dataset,
            controllers=['chesca_official', 'chesca_mesh'],
            episode_steps=None,
            tag='cross_schema_validation',
        )
        table = evaluated.summary.copy()
        table.insert(0, 'dataset', dataset)
        tables.append(table)
    cross_schema_summary = pd.concat(tables, ignore_index=True)
    display(cross_schema_summary)